# Figure 3A: implicit-solvent (PCM) benefit by method, chloroform vs benzene

Per-split percent benefit of adding a PCM solvent correction, for each method, in chloroform and benzene.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import delta22
import paths
import fig3_plots

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
# the published panels use 250 seeded train/test splits
N_SPLITS = 250

In [ ]:
query = delta22.add_composite_columns(delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False))
solutes = sorted(query["solute"].unique())
print(len(query), "rows;", len(solutes), "solutes;", query["sap_nmr_method"].nunique(), "methods")

In [ ]:
fig3a_by_solvent = delta22.fig3a_pcm_benefit_by_solvent(query, ["chloroform", "benzene"],
                                                        n_splits=N_SPLITS, solutes=solutes)

# per-method median percent benefit, then mean +/- std across methods
chloro = fig3a_by_solvent[fig3a_by_solvent["solvent"] == "chloroform"]
benz = fig3a_by_solvent[fig3a_by_solvent["solvent"] == "benzene"]
chloro_by_method = chloro.groupby("sap_nmr_method")["percent_benefit"].median()
benz_by_method = benz.groupby("sap_nmr_method")["percent_benefit"].median()
print(f"PCM benefit in chloroform: {chloro_by_method.mean():+.1f} +/- {chloro_by_method.std():.1f}%"
      f"  (positive = PCM helps)")
print(f"PCM benefit in benzene:    {benz_by_method.mean():+.1f} +/- {benz_by_method.std():.1f}%"
      f"  (negative = PCM hurts)")

In [ ]:
# Fixed method order, grouped by family (ab initio -> conventional DFT -> NMR-specific -> double
# hybrid), each as (method, basis, geometry). This ordering is what gives the panel its family
# structure; sorting by benefit value would scramble it.
FIG3A_COMBOS = [
    ("hf", "pcSseg3", "pbe0_tz"), ("mp2", "pcSseg2", "pbe0_tz"), ("dlpno_mp2", "pcSseg3", "pbe0_tz"),
    ("b3lyp_d3bj", "pcSseg3", "pbe0_tz"), ("pbe0_d3bj", "pcSseg2", "pbe0_tz"),
    ("m062x_d3", "pcSseg3", "pbe0_tz"), ("b97d3_d3bj", "pcSseg3", "pbe0_tz"),
    ("wb97xd", "pcSseg3", "pbe0_tz"), ("tpsstpss_d3bj", "pcSseg3", "pbe0_tz"),
    ("bp86_d3bj", "pcSseg3", "pbe0_tz"), ("blyp_d3bj", "pcSseg3", "pbe0_tz"),
    ("wp04", "pcSseg3", "pbe0_tz"), ("wc04", "pcSseg3", "pbe0_tz"),
    ("dsd_pbep86", "pcSseg3", "pbe0_tz"), ("B2GP_PLYP", "pcSseg3", "pbe0_tz"),
    ("B2PLYP", "pcSseg3", "pbe0_tz"), ("mPW2PLYP", "pcSseg3", "pbe0_tz"),
    ("revdsd_pbep86", "pcSseg3", "pbe0_tz"),
]
FIG3A_COLORS = {"chloroform": "#44AA99", "benzene": "#DDCC77"}   # teal / tan, as published

In [ ]:
fig3_plots.plot_pcm_benefit_with_arrows(fig3a_by_solvent, FIG3A_COMBOS, FIG3A_COLORS,
                                        save_path=figure_path("fig3a_pcm_benefit_1H.png"))